# Collection Counts: Existing + Additional Datasets (All Periods)

Shows document counts for:
- **Existing collections** from the metadata index (`D:/English`) with quality filter (top 50%)
- **Additional datasets** (no quality filter, 100% kept):
  - Newswire (full article text, 1878-1977)
  - FT (full article text, OCR, 1888-2006)
  - Economist (full article text, OCR, 1843-present)
  - NYT (abstracts/snippets only, 1851-2017) *
  - WSJ (abstracts only, 1889-2009) *

\* NYT and WSJ do **not** contain full article text.

In [17]:
import pandas as pd
import json
from pathlib import Path
import sys

try:
    import pyarrow.parquet as pq
    def count_parquet_rows(path):
        return pq.ParquetFile(path).metadata.num_rows
except ImportError:
    def count_parquet_rows(path):
        return len(pd.read_parquet(path, columns=[]))

PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.post_training.config import PERIODS, get_paths

In [18]:
# ---- Paths ----
ADDITIONAL_DATA = PROJECT_ROOT / "Data" / "additional_data"
NEWSWIRE_DIR    = ADDITIONAL_DATA / "newswire"
ECONOMIST_DIR   = ADDITIONAL_DATA / "news_archives" / "Economist"
NYT_DIR         = ADDITIONAL_DATA / "news_archives" / "NYT"
FT_DIR          = ADDITIONAL_DATA / "news_archives" / "FT"
WSJ_PATH        = ADDITIONAL_DATA / "news_archives" / "WSJ" / "wsj_articles.parquet"

# ---- Settings ----
quality_percentile = 50
max_per_collection = 10_000

print("Paths:")
for label, p in [("Newswire", NEWSWIRE_DIR), ("Economist", ECONOMIST_DIR),
                  ("NYT", NYT_DIR), ("FT", FT_DIR), ("WSJ", WSJ_PATH)]:
    print(f"  {label:<12} {p}  (exists: {p.exists()})")

Paths:
  Newswire     C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\newswire  (exists: True)
  Economist    C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\Economist  (exists: True)
  NYT          C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\NYT  (exists: True)
  FT           C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\FT  (exists: True)
  WSJ          C:\Users\danielyoon\Dropbox\hist_LLM\Data\additional_data\news_archives\WSJ\wsj_articles.parquet  (exists: True)


## Pre-compute additional dataset counts per period

This cell counts rows for each additional dataset within each period's year range.  
Newswire (JSON) is the slowest to count — the rest use fast parquet metadata reads.

In [19]:
# Load WSJ year column once (single file covering all years)
wsj_years = pd.read_parquet(WSJ_PATH, columns=["year"]) if WSJ_PATH.exists() else pd.DataFrame()

def count_additional_datasets(start, end):
    """Count rows for each additional dataset within [start, end] year range."""
    counts = {}

    # Newswire: JSON per year, 1878-1977
    nw = 0
    nw_errors = []
    for y in range(max(start, 1878), min(end, 1977) + 1):
        p = NEWSWIRE_DIR / f"{y}_data_clean.json"
        if p.exists():
            try:
                with open(p, "r", encoding="utf-8") as f:
                    nw += len(json.load(f))
            except (json.JSONDecodeError, UnicodeDecodeError) as e:
                nw_errors.append(y)
    if nw_errors:
        print(f"\n    [!] Newswire: skipped corrupted files for years {nw_errors}")
    counts["Newswire"] = nw

    # Economist: parquet per weekly issue, 1843-present
    econ = 0
    for y in range(start, end + 1):
        for f in ECONOMIST_DIR.glob(f"economist_{y}-*.parquet"):
            econ += count_parquet_rows(f)
    counts["Economist"] = econ

    # NYT: parquet per year, 1851-2017
    nyt = 0
    for y in range(max(start, 1851), min(end, 2017) + 1):
        p = NYT_DIR / f"nyt_{y}.parquet"
        if p.exists():
            nyt += count_parquet_rows(p)
    counts["NYT *"] = nyt

    # FT: parquet per year, 1888-2006
    ft = 0
    for y in range(max(start, 1888), min(end, 2006) + 1):
        p = FT_DIR / f"{y}.parquet"
        if p.exists():
            ft += count_parquet_rows(p)
    counts["FT"] = ft

    # WSJ: single file, filter by year
    if len(wsj_years) > 0:
        counts["WSJ *"] = int(((wsj_years["year"] >= start) & (wsj_years["year"] <= end)).sum())
    else:
        counts["WSJ *"] = 0

    return counts


# Pre-compute for all periods
additional_counts = {}
for period_name, (start, end) in PERIODS.items():
    print(f"Counting {period_name} ({start}-{end})...", end=" ", flush=True)
    additional_counts[period_name] = count_additional_datasets(start, end)
    totals = sum(additional_counts[period_name].values())
    print(f"{totals:,} total")

print("\nDone.")

Counting 1678_1849 (1678-1849)... 12,990 total
Counting 1850_1899 (1850-1899)... 2,639,566 total
Counting 1900_1949 (1900-1949)... 10,474,805 total
Counting 1950_1999 (1950-1999)... 
    [!] Newswire: skipped corrupted files for years [1957]
17,222,905 total
Counting 2000_2009 (2000-2009)... 3,478,299 total
Counting 2010_2023 (2010-2023)... 908,611 total

Done.


## Per-period tables

For each period:
- **Existing collections** (from metadata index): quality filtered at top 50%
- **Additional datasets** `[NEW]`: no quality filter (100% kept)
- Sorted by **Raw** count, descending

\* NYT and WSJ only have abstracts/snippets, not full article text.

In [20]:
all_period_summaries = {}  # for the chart later

for period_name, (start, end) in PERIODS.items():
    rows = []

    # --- Existing collections from metadata index ---
    paths = get_paths(period_name)
    metadata_path = paths["metadata_index"]

    if metadata_path.exists():
        meta = pd.read_parquet(metadata_path)
        threshold = meta["predicted_quality"].quantile(quality_percentile / 100)
        meta_filt = meta[meta["predicted_quality"] >= threshold]

        raw_c = meta["collection"].value_counts()
        filt_c = meta_filt["collection"].value_counts()

        for coll in raw_c.index:
            raw = int(raw_c[coll])
            filt = int(filt_c.get(coll, 0))
            rows.append({
                "collection": coll,
                "raw": raw,
                "after_qf": filt,
                "kept_pct": filt / raw * 100 if raw else 0,
                "is_new": False,
            })

    # --- Additional datasets (no QF, 100% kept) ---
    for ds_name, count in additional_counts[period_name].items():
        if count > 0:
            rows.append({
                "collection": f"{ds_name} [NEW]",
                "raw": count,
                "after_qf": count,
                "kept_pct": 100.0,
                "is_new": True,
            })

    if not rows:
        print(f"\n{'='*70}")
        print(f"  {period_name} ({start}-{end})  —  no data")
        continue

    # Sort by raw count descending
    rows.sort(key=lambda r: r["raw"], reverse=True)

    total_raw = sum(r["raw"] for r in rows)
    total_filt = sum(r["after_qf"] for r in rows)
    new_raw = sum(r["raw"] for r in rows if r["is_new"])
    new_filt = sum(r["after_qf"] for r in rows if r["is_new"])

    all_period_summaries[period_name] = {
        "existing_raw": total_raw - new_raw,
        "existing_filt": total_filt - new_filt,
        "new_raw": new_raw,
        "new_filt": new_filt,
    }

    # Print table
    print(f"\n{'='*70}")
    print(f"  {period_name} ({start}-{end})")
    print(f"{'='*70}")
    print(f"{'Collection':<35} {'Raw':>12} {'After QF':>12} {'Kept %':>8}")
    print(f"{'-'*70}")
    for r in rows:
        print(f"{r['collection']:<35} {r['raw']:>12,} {r['after_qf']:>12,} {r['kept_pct']:>7.1f}%")
    print(f"{'-'*70}")
    print(f"{'TOTAL':<35} {total_raw:>12,} {total_filt:>12,} "
          f"{total_filt/total_raw*100:>7.1f}%")
    print(f"  (of which [NEW]):              {new_raw:>12,} {new_filt:>12,}")


  1678_1849 (1678-1849)
Collection                                   Raw     After QF   Kept %
----------------------------------------------------------------------
English-PD                             2,822,461    1,307,585    46.3%
US-PD-Books                              326,640      193,808    59.3%
US-PD-Newspapers                         190,114      114,244    60.1%
Caselaw Access Project                    93,035       83,040    89.3%
LoC-PD-Books                              82,399       62,417    75.7%
French-PD-diverse                         21,822        8,627    39.5%
Economist [NEW]                           12,990       12,990   100.0%
Multilingual-PD                            9,522        4,079    42.8%
Spanish-PD-Books                           4,812        1,754    36.5%
German-PD                                  3,136          855    27.3%
NewZealand-PD-Newspapers                   2,937        2,744    93.4%
Italian-PD                                 1,939    

## Sampling decision per collection (1950-1999)

With the 10K-per-collection cap applied to the combined set.

In [ ]:
# Rebuild rows for 1950-1999 with sampling decision
period_name = "1950_1999"
start, end = PERIODS[period_name]
paths = get_paths(period_name)

rows = []

# Existing
if paths["metadata_index"].exists():
    meta = pd.read_parquet(paths["metadata_index"])
    threshold = meta["predicted_quality"].quantile(quality_percentile / 100)
    meta_filt = meta[meta["predicted_quality"] >= threshold]
    raw_c = meta["collection"].value_counts()
    filt_c = meta_filt["collection"].value_counts()

    for coll in raw_c.index:
        raw = int(raw_c[coll])
        after_qf = int(filt_c.get(coll, 0))
        selected = min(after_qf, max_per_collection)
        rows.append({
            "collection": coll,
            "raw": raw,
            "after_qf": after_qf,
            "selected": selected,
            "action": "sampled" if after_qf > max_per_collection else "all",
        })

# New datasets
for ds_name, count in additional_counts[period_name].items():
    if count > 0:
        selected = min(count, max_per_collection)
        rows.append({
            "collection": f"{ds_name} [NEW]",
            "raw": count,
            "after_qf": count,
            "selected": selected,
            "action": "sampled" if count > max_per_collection else "all",
        })

rows.sort(key=lambda r: r["raw"], reverse=True)

print(f"max_per_collection = {max_per_collection:,}")
print(f"\n{'Collection':<35} {'Raw':>12} {'After QF':>12} {'Selected':>10}  Action")
print("=" * 82)
for r in rows:
    print(f"{r['collection']:<35} {r['raw']:>12,} {r['after_qf']:>12,} "
          f"{r['selected']:>10,}  {r['action']}")
print("=" * 82)
print(f"{'TOTAL':<35} {sum(r['raw'] for r in rows):>12,} "
      f"{sum(r['after_qf'] for r in rows):>12,} "
      f"{sum(r['selected'] for r in rows):>10,}")

## Visual summary across all periods

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    periods = list(all_period_summaries.keys())
    existing_filt = [all_period_summaries[p]["existing_filt"] for p in periods]
    new_filt      = [all_period_summaries[p]["new_filt"] for p in periods]

    x = np.arange(len(periods))
    w = 0.5

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x, existing_filt, w, label="Existing (after QF)", color="#6baed6")
    ax.bar(x, new_filt, w, bottom=existing_filt, label="Additional [NEW] (no QF)", color="#fd8d3c")

    ax.set_xticks(x)
    ax.set_xticklabels([p.replace('_', '-') for p in periods], rotation=30, ha="right")
    ax.set_ylabel("Documents (after QF / no QF)")
    ax.set_title("Available Documents per Period: Existing vs Additional")
    ax.legend()

    # Add count labels
    for i, (e, n) in enumerate(zip(existing_filt, new_filt)):
        total = e + n
        if total > 0:
            ax.text(i, total + total * 0.01, f"{total:,.0f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not available - skipping chart")

## Grand summary table

In [ ]:
print(f"{'Period':<15} {'Existing Raw':>14} {'Existing QF':>14} {'New Raw':>14} {'New (no QF)':>14} {'Combined':>14}")
print("=" * 90)
for p in all_period_summaries:
    s = all_period_summaries[p]
    combined = s["existing_filt"] + s["new_filt"]
    print(f"{p:<15} {s['existing_raw']:>14,} {s['existing_filt']:>14,} "
          f"{s['new_raw']:>14,} {s['new_filt']:>14,} {combined:>14,}")
print("=" * 90)
print(f"{'TOTAL':<15} "
      f"{sum(s['existing_raw'] for s in all_period_summaries.values()):>14,} "
      f"{sum(s['existing_filt'] for s in all_period_summaries.values()):>14,} "
      f"{sum(s['new_raw'] for s in all_period_summaries.values()):>14,} "
      f"{sum(s['new_filt'] for s in all_period_summaries.values()):>14,} "
      f"{sum(s['existing_filt'] + s['new_filt'] for s in all_period_summaries.values()):>14,}")